In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from matplotlib import pyplot
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from nltk.corpus import stopwords
from nltk import word_tokenize
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, classification_report
from transformers import pipeline

try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

def preprocess_pandas(data, columns):
    df_ = pd.DataFrame(columns=columns)
    data['Sentence'] = data['Sentence'].str.lower()
    data['Sentence'] = data['Sentence'].replace(r'[a-zA-Z0-9-_.]+@[a-zA-Z0-9-_.]+', '', regex=True)
    data['Sentence'] = data['Sentence'].replace(r'((25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)(\.|$)){4}', '', regex=True)
    data['Sentence'] = data['Sentence'].str.replace(r'[^\w\s]','')
    data['Sentence'] = data['Sentence'].replace(r'\d', '', regex=True)
    for index, row in data.iterrows():
        word_tokens = word_tokenize(row['Sentence'])
        filtered_sent = [w for w in word_tokens if not w in stopwords.words('english')]
        df_.loc[len(df_)] = {
            "index": row['index'],
            "Class": row['Class'],
            "Sentence": " ".join(filtered_sent)
        }
    return df_

# Define ANN
class Net(nn.Module):
    def __init__ (self,sz):
        super(Net,self).__init__()
        self.fc1 = nn.Linear(sz,64)
        self.fc2 = nn.Linear(64,64)
        self.fc3 = nn.Linear(64,1)

    def forward(self,x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return torch.sigmoid(x)

# Define LSTM Network
class LSTMNet(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(LSTMNet, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (h_n, c_n) = self.lstm(embedded)

        last_hidden = torch.cat((h_n[-2,:,:], h_n[-1,:,:]), dim=1)

        out = self.fc(last_hidden)
        return self.sigmoid(out)

# Helper function for LSTM data preparation
def sentence_to_indices(sentence, vocab):
    return [vocab[w] for w in sentence.split() if w in vocab]

# Define max_len for LSTM padding
max_len = 20

# Initialize the transformer pipeline
classifier = pipeline("sentiment-analysis")


data_1k = pd.read_csv(r"E:\lab1\amazon_cells_labelled.txt", delimiter='\t', header=None)
data_1k.columns = ['Sentence', 'Class']
data_1k['index'] = data_1k.index
columns = ['index', 'Class', 'Sentence']
data_1k = preprocess_pandas(data_1k, columns)


# Split the data into training, validation sets, preserving raw text for LSTM and labels
raw_training_sentences_1k, raw_validation_sentences_1k, training_labels_1k, validation_labels_1k = train_test_split( # split the data into training, validation, and test splits
    data_1k['Sentence'].values.astype('U'),
    data_1k['Class'].values.astype('int32'),
    test_size=0.10,
    random_state=0,
    shuffle=True
)

# Save a copy of raw validation sentences specifically for LSTM
val_texts_for_lstm_1k = raw_validation_sentences_1k.copy()


# Initialize TFIDF vectorizer
word_vectorizer = TfidfVectorizer(analyzer='word', ngram_range=(1,2), max_features=50000, max_df=0.5, use_idf=True, norm='l2')

# Apply TFIDF to the raw text sentences to get vectorized data for the ANN
training_data_tfidf_1k = word_vectorizer.fit_transform(raw_training_sentences_1k)        # transform texts to sparse matrix
training_data_tfidf_1k = training_data_tfidf_1k.todense()                             # convert to dense matrix for Pytorch
vocab_size_1k = len(word_vectorizer.vocabulary_)

validation_data_tfidf_1k = word_vectorizer.transform(raw_validation_sentences_1k)
validation_data_tfidf_1k = validation_data_tfidf_1k.todense()

train_x_tensor_1k = torch.from_numpy(np.array(training_data_tfidf_1k)).type(torch.FloatTensor)
train_y_tensor_1k = torch.from_numpy(np.array(training_labels_1k)).long()
validation_x_tensor_1k = torch.from_numpy(np.array(validation_data_tfidf_1k)).type(torch.FloatTensor)
validation_y_tensor_1k = torch.from_numpy(np.array(validation_labels_1k)).long()


model = Net(vocab_size_1k)
print(model)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

tset = TensorDataset(train_x_tensor_1k, train_y_tensor_1k.float().unsqueeze(1))
tloader = DataLoader(tset, batch_size=32, shuffle=True)
# using 10 epochs to make sure the model will not Overfitting
for epoch in range(10):
  model.train()
  running_loss = 0.0
  for i,(inputs,labels) in enumerate(tloader):
        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()
        running_loss += loss.item()

  print(f"epoch {epoch + 1} avg loss: {running_loss / len(tloader):.4f}")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cuda:0


Net(
  (fc1): Linear(in_features=5145, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=1, bias=True)
)
epoch 1 avg loss: 0.6919
epoch 2 avg loss: 0.6552
epoch 3 avg loss: 0.4958
epoch 4 avg loss: 0.2227
epoch 5 avg loss: 0.0721
epoch 6 avg loss: 0.0274
epoch 7 avg loss: 0.0149
epoch 8 avg loss: 0.0098
epoch 9 avg loss: 0.0071
epoch 10 avg loss: 0.0057


In [ ]:

data_25k = pd.read_csv(r"E:\lab1\amazon_cells_labelled_LARGE_25K.txt", delimiter='\t', header=None)
data_25k.columns = ['Sentence', 'Class']
data_25k['index'] = data_25k.index
columns = ['index', 'Class', 'Sentence']
data_25k = preprocess_pandas(data_25k, columns)

preparing the 25K dataset for ANN


In [ ]:
# Split the 25K data into training, validation sets, preserving raw text for LSTM and labels
raw_training_sentences_25k, raw_validation_sentences_25k, training_labels_25k, validation_labels_25k = train_test_split(
    data_25k['Sentence'].values.astype('U'),
    data_25k['Class'].values.astype('int32'),
    test_size=0.10,
    random_state=0,
    shuffle=True
)

# Save a copy of raw validation sentences specifically for LSTM and Transformer
val_texts_for_lstm_25k = raw_validation_sentences_25k.copy()

# Apply TFIDF to the raw text sentences to get vectorized data for the ANN
word_vectorizer_25k = TfidfVectorizer(analyzer='word', ngram_range=(1,2), max_features=50000, max_df=0.5, use_idf=True, norm='l2')
training_data_tfidf_25k = word_vectorizer_25k.fit_transform(raw_training_sentences_25k)
training_data_tfidf_25k = training_data_tfidf_25k.todense()
vocab_size_25k = len(word_vectorizer_25k.vocabulary_)

validation_data_tfidf_25k = word_vectorizer_25k.transform(raw_validation_sentences_25k)
validation_data_tfidf_25k = validation_data_tfidf_25k.todense()

train_x_tensor_25k = torch.from_numpy(np.array(training_data_tfidf_25k)).type(torch.FloatTensor)
train_y_tensor_25k = torch.from_numpy(np.array(training_labels_25k)).long()
validation_x_tensor_25k = torch.from_numpy(np.array(validation_data_tfidf_25k)).type(torch.FloatTensor)
validation_y_tensor_25k = torch.from_numpy(np.array(validation_labels_25k)).long()

training ANN with 25K data


In [ ]:
model_25k = Net(vocab_size_25k)
print(model_25k)

criterion_25k = nn.BCELoss()
optimizer_25k = optim.Adam(model_25k.parameters())

tset_25k = TensorDataset(train_x_tensor_25k, train_y_tensor_25k.float().unsqueeze(1))
tloader_25k = DataLoader(tset_25k, batch_size=32, shuffle=True)

for epoch in range(10):
  model_25k.train()
  running_loss = 0.0
  for i,(inputs,labels) in enumerate(tloader_25k):
        optimizer_25k.zero_grad()

        outputs = model_25k(inputs)

        loss = criterion_25k(outputs, labels)

        loss.backward()

        optimizer_25k.step()
        running_loss += loss.item()

  print(f"epoch {epoch + 1} avg loss: {running_loss / len(tloader_25k):.4f}")

Net(
  (fc1): Linear(in_features=50000, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=1, bias=True)
)
epoch 1 avg loss: 0.3860
epoch 2 avg loss: 0.1187
epoch 3 avg loss: 0.0275
epoch 4 avg loss: 0.0049
epoch 5 avg loss: 0.0007
epoch 6 avg loss: 0.0002
epoch 7 avg loss: 0.0001
epoch 8 avg loss: 0.0000
epoch 9 avg loss: 0.0000
epoch 10 avg loss: 0.0000


In [ ]:
# in this cell we prepare the data for 1K LSTM and instantiate the model
all_words = " ".join(data_1k['Sentence']).split()
vocab = {word: i+1 for i, word in enumerate(set(all_words))}
vocab_size = len(vocab) + 1

encoded_sentences = [sentence_to_indices(s, vocab) for s in data_1k['Sentence']]
padded_sentences = [s[:max_len] + [0]*(max_len - len(s)) for s in encoded_sentences]

train_x_lstm = torch.LongTensor(padded_sentences)
train_y_lstm = torch.from_numpy(data_1k['Class'].values).float().unsqueeze(1)

model_lstm = LSTMNet(vocab_size, embedding_dim=64, hidden_dim=32)
print(model_lstm)

LSTMNet(
  (embedding): Embedding(1762, 64)
  (lstm): LSTM(64, 32, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


preparing the 25K dataset

In [ ]:
# Prepare 25K data for LSTM
all_words_25k = " ".join(data_25k['Sentence']).split()
vocab_25k = {word: i+1 for i, word in enumerate(set(all_words_25k))}
vocab_size_lstm_25k = len(vocab_25k) + 1

encoded_sentences_25k = [sentence_to_indices(s, vocab_25k) for s in data_25k['Sentence']]
padded_sentences_25k = [s[:max_len] + [0]*(max_len - len(s)) for s in encoded_sentences_25k]

train_x_lstm_25k = torch.LongTensor(padded_sentences_25k)
train_y_lstm_25k = torch.from_numpy(data_25k['Class'].values).float().unsqueeze(1)

model_lstm_25k = LSTMNet(vocab_size_lstm_25k, embedding_dim=64, hidden_dim=32)
print(model_lstm_25k)

LSTMNet(
  (embedding): Embedding(25055, 64)
  (lstm): LSTM(64, 32, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [ ]:
# here is the training of LSTMNet and define Binary Cross-Entropy Loss
lstm_dataset = TensorDataset(train_x_lstm, train_y_lstm)
lstm_loader = DataLoader(lstm_dataset, batch_size=32, shuffle=True)

criterion_lstm = nn.BCELoss() # Binary...Loss
optimizer_lstm = optim.Adam(model_lstm.parameters(), lr=0.001)

for epoch in range(5):
    model_lstm.train()
    total_loss = 0
    for batch_x, batch_y in lstm_loader:
        optimizer_lstm.zero_grad()
        outputs = model_lstm(batch_x)
        loss = criterion_lstm(outputs, batch_y)
        loss.backward()
        optimizer_lstm.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(lstm_loader):.4f}")

Epoch 1, Loss: 0.6888
Epoch 2, Loss: 0.6613
Epoch 3, Loss: 0.6202
Epoch 4, Loss: 0.5431
Epoch 5, Loss: 0.4530


training LSTM with 25K data


In [ ]:
lstm_dataset_25k = TensorDataset(train_x_lstm_25k, train_y_lstm_25k)
lstm_loader_25k = DataLoader(lstm_dataset_25k, batch_size=32, shuffle=True)

criterion_lstm_25k = nn.BCELoss()
optimizer_lstm_25k = optim.Adam(model_lstm_25k.parameters(), lr=0.001)

for epoch in range(5):
    model_lstm_25k.train()
    total_loss = 0
    for batch_x, batch_y in lstm_loader_25k:
        optimizer_lstm_25k.zero_grad()
        outputs = model_lstm_25k(batch_x)
        loss = criterion_lstm_25k(outputs, batch_y)
        loss.backward()
        optimizer_lstm_25k.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(lstm_loader_25k):.4f}")

Epoch 1, Loss: 0.4938
Epoch 2, Loss: 0.3392
Epoch 3, Loss: 0.2656
Epoch 4, Loss: 0.2075
Epoch 5, Loss: 0.1597


In [ ]:
# we use a pretrained transformer model to recognize feelings in sentences
transformer_results = classifier(list(val_texts_for_lstm_1k))
trans_preds = [1 if res['label'] == 'POSITIVE' else 0 for res in transformer_results]
trans_acc = accuracy_score(validation_labels_1k, trans_preds)
print(f"Transformer Accuracy: {trans_acc * 100:.2f}%")

Transformer Accuracy: 86.00%


evaluating Transformer with 25K data

In [ ]:
transformer_results_25k = classifier(list(val_texts_for_lstm_25k))
trans_preds_25k = [1 if res['label'] == 'POSITIVE' else 0 for res in transformer_results_25k]
trans_acc_25k = accuracy_score(validation_labels_25k, trans_preds_25k)
print(f"Transformer Accuracy (25K data): {trans_acc_25k * 100:.2f}%")

Transformer Accuracy (25K data): 80.36%


In [ ]:
model_lstm.eval()
with torch.no_grad():
    val_encoded_1k = [sentence_to_indices(s, vocab) for s in val_texts_for_lstm_1k]
    val_padded_1k = [s[:max_len] + [0]*(max_len - len(s)) for s in val_encoded_1k]
    val_x_lstm_tensor_1k = torch.LongTensor(val_padded_1k)

    lstm_out_1k = model_lstm(val_x_lstm_tensor_1k)
    lstm_preds_1k = (lstm_out_1k > 0.5).float().flatten()

    correct_1k = (lstm_preds_1k == torch.tensor(validation_labels_1k).float()).sum().item()
    print(f"LSTM Accuracy: {correct_1k / len(validation_labels_1k) * 100:.2f}%")

LSTM Accuracy: 93.00%


evaluating LSTM  with 25k


In [ ]:
model_lstm_25k.eval()
with torch.no_grad():
    val_encoded_25k = [sentence_to_indices(s, vocab_25k) for s in val_texts_for_lstm_25k]
    val_padded_25k = [s[:max_len] + [0]*(max_len - len(s)) for s in val_encoded_25k]
    val_x_lstm_tensor_25k = torch.LongTensor(val_padded_25k)

    lstm_out_25k = model_lstm_25k(val_x_lstm_tensor_25k)
    lstm_preds_25k = (lstm_out_25k > 0.5).float().flatten()

    correct_25k = (lstm_preds_25k == torch.tensor(validation_labels_25k).float()).sum().item()
    print(f"LSTM Accuracy (25K data): {correct_25k / len(validation_labels_25k) * 100:.2f}%")

LSTM Accuracy (25K data): 96.24%


In [ ]:
model.eval()
with torch.no_grad():
    ann_out = model(validation_x_tensor_1k)
    ann_preds = (ann_out > 0.5).float().flatten()
    ann_acc = (ann_preds == validation_y_tensor_1k.float()).float().mean()
    print(f"Simple ANN Accuracy: {ann_acc.item() * 100:.2f}%")

Simple ANN Accuracy: 84.00%


evaluating Simple ANN for 25k data



In [ ]:
model_25k.eval()
with torch.no_grad():
    ann_out_25k = model_25k(validation_x_tensor_25k)
    ann_preds_25k = (ann_out_25k > 0.5).float().flatten()
    ann_acc_25k = (ann_preds_25k == validation_y_tensor_25k.float()).float().mean()
    print(f"Simple ANN Accuracy (25K data): {ann_acc_25k.item() * 100:.2f}%")

Simple ANN Accuracy (25K data): 84.36%




Performance and Scenarios:

LSTM 25K outperformed other models with its 96,24% accuracy. This means it has strong ability to capture sequential dependencies in text when trained on a pretty large dataset.
The pre-trained transformer showed strong performance at 80,36% it was slightly lower than the LSTM, its performance is good. Transformers is better than others in scenarios where understanding global context and long-range dependencies is crucial and where computational resources for pre-trained models are available.
Simple ANN 25K achieved 84,36%. It performed better than the Transformer which is strange because ANNs are usually good for simpler more direct classification tasks but when it trained on larger datasets 25K it can still achieve decent performance.

Scenario Preferences

For high accuracy on sequential data: LSTM is a good choice

For general-purpose understanding: Transformer is the best choice

For simpler, faster models on well-structured feature sets: Simple ANN is good

complexity, accuracy and efficiency

Complexity:
Transformer is the most complex due to its multi-head attention mechanisms and usually larger number of parameters

LSTM is not that complex. Recurrent nature allows it to process sequences but lacks the global context of attention mechanisms.

Simple ANN is the least complex, a feed-forward network, often means faster training and inference.

Accuracy:
On the 25k dataset LSTM  > Simple ANN > Transformer

The LSTM model trained on the larger 25K dataset achieved the highest accuracy demonstrating its capability for sentiment analysis with sufficient data.

While the transformer isa a good choice for this specific task and dataset a well-tuned LSTM with more training data could outperform a general pre-trained Transformer.

Efficiency:
Simple ANN is generally the most efficient in terms of training time per epoch and inference speed due to its simpler architecture and fewer operations per input

LSTM is less efficient than ANN due to sequential processing and internal gates

Transformer can be very computationally intensive during pre-training. Inference with pre-trained models can be fast but fine-tuning requires significant resources.

What insights did you obtain concerning data amount to train?

ANN and LSTM: these models especially when trained from zore they need a lot of data.

Transformer is more data efficient for specific tasks if a relevant pre-trained model is available.

Embedding utilized

ANN Used TF-IDF vectorization. TF-IDF captures word importance based on frequency but doesn't inherently understand semantic relationships or word order. This limitation likely contributed to its lower performance compared to models using more sophisticated embeddings

LSTM used an nn.Embedding layer which learns dense vector representations  for words during training. These embeddings capture semantic meaning and relationships between words more effectively than TF-IDF contributing to LSTM's superior performance on sequential data.


Transformer is utilizes sophisticated contextual embeddings that are learned during extensive pre-training. These embeddings are highly rich.

Architectural choices made?

Simple ANN: A feed-forward network that processes a flattened input. It lacks mechanisms to handle sequence order or long range dependencies, making it less suitable for complex natural language understanding where word order is crucial.

LSTM is a recurrent neural network designed to process sequential data. Its gating mechanisms allow it to selectively remember or forget information over time making it effective for capturing dependencies in sentences

Transformer relies on self attention mechanisms to weigh the importance of different words in a sequence relative to each other irrespective of their distance. This allows it to capture global dependencies and parallelize computation efficiently overcoming the sequential bottlenecks of LSTMs. Its architecture is well-suited for understanding complex language structures and contexts.
